# Gemini quality test: ingredient lines -> name, amount, unit

Sends the same few batches with every config in `TEST_CONFIGS` (model x thinking level) of distinct ingredient lines (mixed languages) to Gemini and reports
everything needed to plan the full run: input / thinking / output tokens, latency, missing lines,
automatic quality flags, a side-by-side review, and how many days the free quota needs for all lines.

- The database is opened **read-only**, so this never blocks the crawler and changes nothing.
- Every request is appended to `data.nosync/gemini_usage.jsonl`, so the requests used today can be counted
  (the API does not report remaining quota). Each request uses 1 of the 500/day of its key x model.

In [1]:
import json
import math
import random
import sqlite3
import time
from collections import Counter, defaultdict
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

from google import genai
from google.genai import errors, types

DB_PATH = "data.nosync/recipes.db"
USAGE_LOG = "data.nosync/gemini_usage.jsonl"  # one line per request (git-ignored)
RESULTS_PATH = "data.nosync/quality_test_results.json"

# (model, thinking level) pairs to compare on the same lines. Thinking: "minimal", "low", "medium", "high" or None.
# Each pair costs N_BATCHES requests. Add ("gemini-3.1-flash-lite", "minimal") to test the second model too.
TEST_CONFIGS = [
    ("gemini-3.5-flash-lite", "minimal"),
    ("gemini-3.5-flash-lite", "low"),
]
PROD_CONFIG = TEST_CONFIGS[0]  # what the real run would use, for the projection at the end
PROD_MODELS = [PROD_CONFIG[0]]  # models whose quota the real run can use
BATCH_SIZE = 500  # lines per request
N_BATCHES = 2  # requests per config; they alternate between the keys so every key gets tested
MAX_OUTPUT_TOKENS = 32768  # thinking counts towards this too
label = lambda config: f"{config[0]} / thinking {config[1]}"

# Free tier limits per key (= per Google Cloud project) and model. Check them in AI Studio.
RPD = 500
RPM = 15
TPM = 250_000
QUOTA_PROJECTS = 2  # different Google Cloud projects behind the keys; keys of the same project share one quota
QUOTA_TZ = ZoneInfo("America/Los_Angeles")  # daily quotas reset at midnight Pacific time

DEADLINE = datetime(2026, 11, 2, 8, 0).astimezone()  # Monday morning, local time

# Paid prices in USD per 1M tokens, only for the cost estimate. Thinking is billed as output. None = skip.
PRICE_IN = None
PRICE_OUT = None

with open("gemini_keys.txt") as f:
    API_KEYS = [line.strip() for line in f if line.strip() and not line.startswith("#")]
API_KEYS = list(dict.fromkeys(API_KEYS))
clients = [genai.Client(api_key=key) for key in API_KEYS]
print(f"{len(clients)} keys")

# Read-only: never blocks the crawler and cannot change anything.
con = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, timeout=60)

2 keys


In [2]:
UNITS = ["g", "kg", "mg", "ml", "l", "tsp", "tbsp", "cup", "oz", "lb", "fl oz", "pinch", "dash", "piece", "clove",
         "slice", "can", "package", "bunch", "sprig", "stalk", "leaf", "handful", "drop"]

PROMPT = f"""You turn recipe ingredient lines into structured data.

The input is a JSON array of [index, line]. The lines can be in any language.
For every line return one or more objects {{"i": index, "en": name, "a": amount, "u": unit}}:
- en: the ingredient name TRANSLATED INTO ENGLISH, lowercase, singular. Always English, also when the line is a
  single word: never copy a word from the input unless it is also the English word ("avocado", "curry").
  Only the ingredient: no quantity, unit, brand, size or notes.
  Drop preparation and state words (fresh, frozen, dried, ground, chopped, minced, sliced, grated, peeled, ...),
  keep words that make it a different ingredient ("brown sugar", "egg white", "soy sauce", "coconut milk").
- a: the amount as a number. Fractions as decimals ("1½" -> 1.5), ranges as the first number ("2-3" -> 2), null if none.
- u: the unit in English, singular. Use one of {", ".join(UNITS)} if it fits,
  otherwise a short English word. null if there is none; plain counts ("3 eggs") have no unit.
- Alternatives ("butter or margarine"): only the first one.
- Several ingredients in one line ("salt and pepper"): one object each, with the same i.
- Units: "el", "ek", "msk", "spl", "c.s.", "ст. л." -> tbsp; "tl", "tsk", "c.c.", "ч. л." -> tsp; "gr" -> g;
  "dag" -> 10 g (a=amount*10, u="g"); "stk", "ks", "db", "szt", "шт" and similar counting words -> piece.
- Not an ingredient (headings like "for the sauce:", equipment, tips): {{"i": index, "en": null, "a": null, "u": null}}.
Every index must appear in the output.

Example input:  [[0, "2 cups chopped onions"], [1, "salz und pfeffer"], [2, "für die soße:"], [3, "1 el sojasauce"],
                 [4, "3 eier"], [5, "1 cebolla"], [6, "czosnek"], [7, "1 kg брашно"], [8, "水 2大匙"], [9, "1 ks vejce"]]
Example output: [{{"i":0,"en":"onion","a":2,"u":"cup"}},{{"i":1,"en":"salt","a":null,"u":null}},{{"i":1,"en":"pepper","a":null,"u":null}},{{"i":2,"en":null,"a":null,"u":null}},{{"i":3,"en":"soy sauce","a":1,"u":"tbsp"}},{{"i":4,"en":"egg","a":3,"u":null}},{{"i":5,"en":"onion","a":1,"u":"piece"}},{{"i":6,"en":"garlic","a":null,"u":null}},{{"i":7,"en":"flour","a":1,"u":"kg"}},{{"i":8,"en":"water","a":2,"u":"tbsp"}},{{"i":9,"en":"egg","a":1,"u":"piece"}}]

Input:
"""

RESPONSE_SCHEMA = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "i": {"type": "integer"},
            "en": {"type": ["string", "null"]},
            "a": {"type": ["number", "null"]},
            "u": {"type": ["string", "null"]},
        },
        "required": ["i", "en", "a", "u"],
    },
}

# Tokens of the instructions alone = fixed overhead paid on every request.
PROMPT_TOKENS = clients[0].models.count_tokens(model=TEST_CONFIGS[0][0], contents=PROMPT).total_tokens
print(f"prompt overhead: {PROMPT_TOKENS} tokens per request")

prompt overhead: 786 tokens per request


In [3]:
def norm(line):
    """The key lines are deduplicated on: lowercase, whitespace collapsed."""
    return " ".join(str(line).split()).lower()


# ~0.5% of all recipes is plenty to draw the sample from.
random.seed(42)
by_lang = defaultdict(set)
for lang, ingredients in con.execute("SELECT language, ingredients FROM recipes WHERE rowid % 211 = 7"):
    lang = (lang or "?")[:2].lower()
    for line in json.loads(ingredients):
        if line := norm(line):
            by_lang[lang].add(line)

# Take lines from every language in turn, so rare languages get checked too, then mix them like the real run.
pools = {lang: random.sample(sorted(lines), len(lines)) for lang, lines in by_lang.items()}
sample, seen = [], set()
while len(sample) < BATCH_SIZE * N_BATCHES and any(pools.values()):
    for lang in sorted(pools, key=lambda l: -len(by_lang[l])):
        if pools[lang] and len(sample) < BATCH_SIZE * N_BATCHES:
            line = pools[lang].pop()
            if line not in seen:
                seen.add(line)
                sample.append((lang, line))
random.shuffle(sample)
print(f"{len(sample)} lines from {len(by_lang)} languages:", dict(Counter(lang for lang, _ in sample).most_common()))

# All distinct lines in the database, for the projection. Takes ~15 s.
t = time.time()
TOTAL_LINES = con.execute(
    "SELECT COUNT(DISTINCT lower(trim(j.value))) FROM recipes r, json_each(r.ingredients) j"
).fetchone()[0]
print(f"{TOTAL_LINES:,} distinct lines in the database ({time.time() - t:.0f} s)")

1000 lines from 38 languages: {'en': 28, 'vi': 27, 'lt': 27, 'tr': 27, 'de': 27, 'is': 27, 'ko': 27, 'pt': 27, 'fi': 27, 'da': 27, 'es': 27, 'uk': 27, 'ru': 27, 'ms': 27, 'id': 27, 'zh': 27, 'el': 27, 'fr': 27, 'nl': 27, 'cs': 27, 'sk': 27, 'no': 27, 'ja': 27, 'it': 27, 'th': 27, 'he': 27, 'ar': 27, 'pl': 27, 'sl': 27, 'bg': 27, 'sr': 27, 'sv': 27, 'ro': 27, 'lv': 27, 'hu': 27, 'et': 27, 'sq': 20, 'fa': 7}
2,066,943 distinct lines in the database (16 s)


In [4]:
# "model is experiencing high demand" (503) and other server errors: wait 15, 30, 60, 120, 240 s and try again.
RETRY_CODES = {500, 502, 503, 504}
MAX_RETRIES = 5
RETRY_WAIT = 15


def quota_day(ts=None):
    return datetime.fromtimestamp(ts or time.time(), QUOTA_TZ).date().isoformat()


def log_usage(rec):
    with open(USAGE_LOG, "a") as f:
        f.write(json.dumps(rec) + "\n")


def used_today():
    """Requests per (key no., model) since the last quota reset. Failed requests count too, to be safe."""
    used = Counter()
    try:
        with open(USAGE_LOG) as f:
            for line in f:
                rec = json.loads(line)
                if rec["day"] == quota_day():
                    used[(rec["key"], rec["model"])] += 1
    except FileNotFoundError:
        pass
    return used


def call(key_no, model, thinking, lines):
    """One request. Returns (parsed JSON or None, record with every number the API reports)."""
    payload = json.dumps([[i, line] for i, line in enumerate(lines)], ensure_ascii=False)
    config = dict(
        temperature=0,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        response_mime_type="application/json",
        response_json_schema=RESPONSE_SCHEMA,
    )
    if thinking:
        config["thinking_config"] = types.ThinkingConfig(thinking_level=thinking)
    rec = {"ts": time.time(), "day": quota_day(), "key": key_no, "model": model, "thinking": thinking, "lines": len(lines), "chars": len(payload)}
    for attempt in range(MAX_RETRIES + 1):
        t = time.time()
        try:
            response = clients[key_no].models.generate_content(
                model=model, contents=PROMPT + payload, config=types.GenerateContentConfig(**config)
            )
            break
        except errors.APIError as e:
            rec.update(latency=time.time() - t, error=f"{e.code} {e.message}"[:300], retries=attempt)
            if e.code not in RETRY_CODES or attempt == MAX_RETRIES:
                log_usage(rec)
                return None, rec
            log_usage(rec)  # logged too, in case overloaded requests count towards the daily quota
            wait = RETRY_WAIT * 2 ** attempt
            print(f"   {e.code} on key{key_no + 1}/{model}, retry {attempt + 1}/{MAX_RETRIES} in {wait} s")
            time.sleep(wait)
    rec.pop("error", None)
    rec.update(ts=time.time(), latency=time.time() - t, retries=attempt)
    usage = response.usage_metadata
    finish = response.candidates[0].finish_reason if response.candidates else None
    rec.update(
        prompt_tokens=usage.prompt_token_count or 0,
        thoughts_tokens=usage.thoughts_token_count or 0,
        output_tokens=usage.candidates_token_count or 0,
        cached_tokens=usage.cached_content_token_count or 0,
        total_tokens=usage.total_token_count or 0,
        finish=getattr(finish, "name", finish),
    )
    try:
        result = json.loads(response.text)
    except (json.JSONDecodeError, TypeError) as e:  # e.g. cut off at MAX_TOKENS
        rec["error"] = f"bad json: {e}"[:300]
        result = None
    log_usage(rec)
    return result, rec


def parse(lines, result):
    """{line index: [(name, amount, unit), ...]} and the number of items with an invalid index. None if the request failed."""
    if result is None:
        return None, 0
    out, bad_items = defaultdict(list), 0
    for item in result:
        i = item.get("i")
        if isinstance(i, int) and 0 <= i < len(lines):
            out[i].append((item.get("en"), item.get("a"), item.get("u")))
        else:
            bad_items += 1
    return dict(out), bad_items


print("used today (key no., model):", dict(used_today()) or "nothing")

used today (key no., model): {(0, 'gemini-3.5-flash-lite'): 2, (1, 'gemini-3.5-flash-lite'): 2}


In [5]:
# Sends N_BATCHES requests per config, all configs on the same lines. Sequential, RPM kept per key x model.
batches = [sample[i:i + BATCH_SIZE] for i in range(0, len(sample), BATCH_SIZE)]
runs = []
next_at = defaultdict(float)
for config in TEST_CONFIGS:
    model, thinking = config
    for b, batch in enumerate(batches):
        key_no = b % len(clients)
        time.sleep(max(0, next_at[key_no, model] - time.monotonic()))
        next_at[key_no, model] = time.monotonic() + 1.05 * 60 / RPM
        lines = [line for _, line in batch]
        result, rec = call(key_no, model, thinking, lines)
        parsed, bad_items = parse(lines, result)
        rec.update(answered=len(parsed or {}), bad_items=bad_items, items=sum(map(len, (parsed or {}).values())))
        runs.append({"config": label(config), "batch": batch, "parsed": parsed, "rec": rec})
        print(
            f"{label(config)} key{key_no + 1} batch {b}: {rec['latency']:.1f} s, "
            f"in {rec.get('prompt_tokens')} / think {rec.get('thoughts_tokens')} / out {rec.get('output_tokens')} tokens, "
            f"finish {rec.get('finish')}, answered {len(parsed or {})}/{len(lines)} lines {rec.get('error', '')}"
        )
        if rec.get("error", "").startswith("429"):
            print("quota exceeded, skipping the rest of this config")
            break

with open(RESULTS_PATH, "w") as f:
    json.dump(runs, f, ensure_ascii=False, indent=1)

gemini-3.5-flash-lite / thinking minimal key1 batch 0: 57.5 s, in 8616 / think 0 / out 13574 tokens, finish STOP, answered 500/500 lines 
gemini-3.5-flash-lite / thinking minimal key2 batch 1: 51.8 s, in 8390 / think 0 / out 14753 tokens, finish STOP, answered 500/500 lines 
gemini-3.5-flash-lite / thinking low key1 batch 0: 82.6 s, in 8616 / think 0 / out 15310 tokens, finish STOP, answered 500/500 lines 
gemini-3.5-flash-lite / thinking low key2 batch 1: 42.4 s, in 8390 / think 0 / out 14803 tokens, finish STOP, answered 500/500 lines 


In [6]:
def stats(recs):
    ok = [r for r in recs if "prompt_tokens" in r]
    s = lambda k: sum(r.get(k, 0) for r in ok)
    lines, answered = s("lines"), s("answered")
    print(f"requests:  {len(recs)} sent, {len(ok)} answered, {sum('error' in r for r in recs)} with errors, "
          f"{sum(r.get('finish') not in (None, 'STOP') for r in ok)} not finished with STOP, "
          f"{sum(r.get('retries', 0) for r in recs)} retries after server errors")
    for r in recs:
        if "error" in r:
            print("   error:", r["error"])
    if not ok:
        return None
    print(f"tokens:    input {s('prompt_tokens'):,}  thinking {s('thoughts_tokens'):,}  output {s('output_tokens'):,}  "
          f"cached {s('cached_tokens'):,}  total {s('total_tokens'):,}")
    print(f"           per request: input {s('prompt_tokens') / len(ok):,.0f} (max {max(r['prompt_tokens'] for r in ok):,}), "
          f"thinking {s('thoughts_tokens') / len(ok):,.0f}, output {s('output_tokens') / len(ok):,.0f} "
          f"(max {max(r['output_tokens'] + r['thoughts_tokens'] for r in ok):,} of {MAX_OUTPUT_TOKENS:,} incl. thinking)")
    per_line = {
        "in": (s("prompt_tokens") - PROMPT_TOKENS * len(ok)) / lines,
        "think": s("thoughts_tokens") / lines,
        "out": s("output_tokens") / lines,
    }
    print(f"           per line (without prompt overhead): input {per_line['in']:.1f}, thinking {per_line['think']:.1f}, "
          f"output {per_line['out']:.1f}")
    print(f"latency:   avg {s('latency') / len(ok):.1f} s, max {max(r['latency'] for r in ok):.1f} s")
    print(f"lines:     {lines:,} sent, {answered:,} answered ({answered / lines:.1%}), {lines - answered} missing, "
          f"{s('bad_items')} items with invalid index, {s('items') / max(answered, 1):.2f} items per line")
    peak = max(r["prompt_tokens"] for r in ok) * RPM
    print(f"TPM check: {RPM} requests/min x max input = {peak:,} tokens/min vs limit {TPM:,} -> "
          f"{'ok' if peak <= TPM else f'only {TPM // max(r['prompt_tokens'] for r in ok)} requests/min'}")
    return {"per_line": per_line, "answered_per_request": answered / len(ok),
            "latency": s("latency") / len(ok)}


summary = {}
for config in map(label, TEST_CONFIGS):
    print(f"\n=== {config} ===")
    summary[config] = stats([r["rec"] for r in runs if r["config"] == config])


=== gemini-3.5-flash-lite / thinking minimal ===
requests:  2 sent, 2 answered, 0 with errors, 0 not finished with STOP, 0 retries after server errors
tokens:    input 17,006  thinking 0  output 28,327  cached 0  total 45,333
           per request: input 8,503 (max 8,616), thinking 0, output 14,164 (max 14,753 of 32,768 incl. thinking)
           per line (without prompt overhead): input 15.4, thinking 0.0, output 28.3
latency:   avg 54.7 s, max 57.5 s
lines:     1,000 sent, 1,000 answered (100.0%), 0 missing, 0 items with invalid index, 1.04 items per line
TPM check: 15 requests/min x max input = 129,240 tokens/min vs limit 250,000 -> ok

=== gemini-3.5-flash-lite / thinking low ===
requests:  2 sent, 2 answered, 0 with errors, 0 not finished with STOP, 0 retries after server errors
tokens:    input 17,006  thinking 0  output 30,113  cached 0  total 47,119
           per request: input 8,503 (max 8,616), thinking 0, output 15,056 (max 15,310 of 32,768 incl. thinking)
           per 

In [7]:
# Automatic red flags. They are hints, not proof: look at the examples.
has_digit = lambda text: any(ch.isdigit() for ch in text)


def untranslated(lang, line, name):
    """Name not in English: non-Latin script, or copied word for word from a non-English line.
    Misses untranslated words that were changed a bit, and flags a few real loanwords ("curry" in a German line)."""
    if not name.isascii():
        return True
    return lang != "en" and f" {name} " in f" {' '.join(ch if ch.isalnum() else ' ' for ch in line)} "


for config in map(label, TEST_CONFIGS):
    flags, examples = Counter(), defaultdict(list)
    named = untrans = 0

    def flag(name, line, entries):
        flags[name] += 1
        if len(examples[name]) < 8:
            examples[name].append(f"{line[:70]!r} -> {entries}")

    for run in (r for r in runs if r["config"] == config and r["parsed"] is not None):
        for i, (lang, line) in enumerate(run["batch"]):
            entries = run["parsed"].get(i)
            if entries is None:
                flag("line missing from answer", line, None)
                continue
            for n, a, u in entries:
                if n is None:
                    if has_digit(line):
                        flag("marked 'not an ingredient' but has a number", line, entries)
                    continue
                if lang != "en":
                    named += 1
                    if untranslated(lang, line, n):
                        untrans += 1
                        flag("NOT TRANSLATED", line, entries)
                if has_digit(n):
                    flag("number in name", line, entries)
                if n != n.lower() or len(n) > 40:
                    flag("name not lowercase or very long", line, entries)
                if u is not None and u not in UNITS:
                    flag("unit outside the list", line, entries)
                if a is None and u is not None:
                    flag("unit without amount", line, entries)
                if a is None and has_digit(line):
                    flag("no amount although the line has a number", line, entries)

    print(f"\n=== {config} ===")
    print(f"untranslated: {untrans}/{named} names from non-English lines = {untrans / max(named, 1):.1%}")
    if summary.get(config):
        summary[config]["untranslated"] = untrans / max(named, 1)
    for name, count in flags.most_common():
        print(f"{count:5}  {name}")
        for ex in examples[name]:
            print("         ", ex)


=== gemini-3.5-flash-lite / thinking minimal ===
untranslated: 1/992 names from non-English lines = 0.1%
   97  unit outside the list
          '1 bát con bột chiên xù' -> [('bread crumbs', 1, 'bowl')]
          '0,5 potte frisk koriander' -> [('coriander', 0.5, 'pot')]
          'たらこ １腹' -> [('salted cod roe', 1, 'belly')]
          'vino bianco 10 cl' -> [('white wine', 10, 'cl')]
          'łyżka octu winnego lub 3 łyżki soku z cytryny' -> [('wine vinegar', 1, 'spoon'), ('lemon juice', 3, 'spoon')]
          'łyżka octu winnego lub 3 łyżki soku z cytryny' -> [('wine vinegar', 1, 'spoon'), ('lemon juice', 3, 'spoon')]
          "15 cl d'eau" -> [('water', 15, 'cl')]
          '2 colheres (sopa) de pasta de amendoim' -> [('peanut butter', 2, 'spoon')]
    9  unit without amount
          'pak kripë' -> [('salt', None, 'package')]
          'viilud kurk' -> [('cucumber', None, 'slice')]
          'viilud redis' -> [('radish', None, 'slice')]
          'pak lëng limoni' -> [('lemon jui

In [8]:
# Side by side, per language. Every config's answer under the line.
PER_LANGUAGE = 12
answers = defaultdict(dict)
for run in runs:
    for i, (lang, line) in enumerate(run["batch"]):
        answers[lang, line][run["config"]] = "REQUEST FAILED" if run["parsed"] is None else run["parsed"].get(i)

shown = Counter()
for (lang, line), by_config in sorted(answers.items()):
    if shown[lang] >= PER_LANGUAGE:
        continue
    if not shown[lang]:
        print(f"\n##### {lang}")
    shown[lang] += 1
    print(line[:100])
    for config, entries in by_config.items():
        if entries is None:
            text = "MISSING"
        elif isinstance(entries, str):
            text = entries
        else:
            text = "; ".join(f"{n} | {a} | {u}" for n, a, u in entries)
        print(f"    {config if len(TEST_CONFIGS) > 1 else '->'}: {text}")


##### ar
1 ملعقة كبيرة سكر بودرة (للديكور)
    gemini-3.5-flash-lite / thinking minimal: powdered sugar | 1 | tbsp
    gemini-3.5-flash-lite / thinking low: powdered sugar | 1 | tbsp
2 ملعقة كبيرة زيت زيتون
    gemini-3.5-flash-lite / thinking minimal: olive oil | 2 | spoon
    gemini-3.5-flash-lite / thinking low: olive oil | 2 | tbsp
2 ملعقة كبيرة شبت مفروم
    gemini-3.5-flash-lite / thinking minimal: dill | 2 | tbsp
    gemini-3.5-flash-lite / thinking low: dill | 2 | tbsp
3 ملاعق كبيرة (45 مل) صلصة طماطم
    gemini-3.5-flash-lite / thinking minimal: tomato paste | 3 | spoon
    gemini-3.5-flash-lite / thinking low: tomato paste | 3 | tbsp
الباذنجان 1 حبة مقشر ومقطع
    gemini-3.5-flash-lite / thinking minimal: eggplant | 1 | piece
    gemini-3.5-flash-lite / thinking low: eggplant | 1 | piece
البصل 1 حبة صغير الحجم / مفروم ناعم
    gemini-3.5-flash-lite / thinking minimal: onion | 1 | piece
    gemini-3.5-flash-lite / thinking low: onion | 1 | piece
البطاطس 1 حبة صغيرة الحجم / مف

In [9]:
# Can we finish all distinct lines before the deadline on the free tier?
# Comparison of all configs first.
for config, m in summary.items():
    if m:
        print(f"{config}: untranslated {m.get('untranslated', float('nan')):.1%}, tokens per line in {m['per_line']['in']:.1f} / "
              f"think {m['per_line']['think']:.1f} / out {m['per_line']['out']:.1f}, {m['latency']:.0f} s per request")
print()

m = summary.get(label(PROD_CONFIG))
assert m, f"no successful requests for {label(PROD_CONFIG)}"
lanes = [(k, mod) for k in range(QUOTA_PROJECTS) for mod in PROD_MODELS]  # used_today() is per key, so put one key per project first in the file

# Missing lines have to be sent again, so count only the answered ones.
lines_per_request = m["answered_per_request"]
requests_needed = math.ceil(TOTAL_LINES / lines_per_request)
tok_in = TOTAL_LINES * m["per_line"]["in"] + requests_needed * PROMPT_TOKENS
tok_think = TOTAL_LINES * m["per_line"]["think"]
tok_out = TOTAL_LINES * m["per_line"]["out"]
print(f"{TOTAL_LINES:,} lines / {lines_per_request:.0f} answered per request = {requests_needed:,} requests")
print(f"tokens for everything: input {tok_in / 1e6:.1f}M, thinking {tok_think / 1e6:.1f}M, output {tok_out / 1e6:.1f}M")
if PRICE_IN is not None and PRICE_OUT is not None:
    cost = tok_in / 1e6 * PRICE_IN + (tok_think + tok_out) / 1e6 * PRICE_OUT
    print(f"paid instead: ${cost:,.0f} (batch API: ${cost / 2:,.0f})")

# Quota windows (Pacific days) between now and the deadline. A window only helps as far as there is time left in it.
now = datetime.now(QUOTA_TZ)
used = used_today()
seconds_per_request = max(m["latency"], 60 / RPM)  # one request at a time per lane
print(f"\none lane sends {RPD} requests in {RPD * seconds_per_request / 3600:.1f} h "
      f"(one at a time, {m['latency']:.0f} s each; with parallel requests at the RPM limit {RPD * 60 / RPM / 60:.0f} min)")
capacity, day = 0, now
while day < DEADLINE:
    day_end = min(datetime.combine(day.date() + timedelta(days=1), datetime.min.time(), QUOTA_TZ), DEADLINE)
    hours = (day_end - day).total_seconds() / 3600
    window = sum(min(RPD - used[lane] if day.date() == now.date() else RPD, hours * 3600 * RPM / 60) for lane in lanes)
    capacity += window
    print(f"  {day.date()} (Pacific): {window:,.0f} requests ({hours:.1f} h left in this window)")
    day = day_end

print(f"\nneeded {requests_needed:,} requests, available until {DEADLINE:%a %d.%m. %H:%M}: {capacity:,.0f} "
      f"with {len(lanes)} lanes ({QUOTA_PROJECTS} projects x {PROD_MODELS})")
print(f"full days of quota needed: {requests_needed / (RPD * len(lanes)):.1f}  ->  "
      f"{'fits' if capacity >= requests_needed else f'short by {requests_needed - capacity:,.0f} requests'}")

gemini-3.5-flash-lite / thinking minimal: untranslated 0.1%, tokens per line in 15.4 / think 0.0 / out 28.3, 55 s per request
gemini-3.5-flash-lite / thinking low: untranslated 0.0%, tokens per line in 15.4 / think 0.0 / out 30.1, 63 s per request

2,066,943 lines / 500 answered per request = 4,134 requests
tokens for everything: input 35.2M, thinking 0.0M, output 58.6M

one lane sends 500 requests in 7.6 h (one at a time, 55 s each; with parallel requests at the RPM limit 33 min)
  2026-10-29 (Pacific): 992 requests (14.7 h left in this window)
  2026-10-30 (Pacific): 1,000 requests (24.0 h left in this window)
  2026-10-31 (Pacific): 1,000 requests (24.0 h left in this window)
  2026-11-01 (Pacific): 1,000 requests (24.0 h left in this window)

needed 4,134 requests, available until Mon 02.11. 08:00: 3,992 with 2 lanes (2 projects x ['gemini-3.5-flash-lite'])
full days of quota needed: 4.1  ->  short by 142 requests
